# 08 - Profile and tomogram-slice plot routines

This notebook confirms the curve and slice plotting routines of `Ploter` by feeding them controlled cubes and reading the PNGs they write back into the notebook for inspection. We exercise `plot_profiles` (per-pixel overlay of GT, prediction and Gaussian components), `plot_tomogram_slice` (range / azimuth cross-sections) and `plot_elevation_intensity_slice` (a horizontal plane).

Modules exercised: `pipelines.inference_pipeline.plots.Ploter`. Figures are written to a temporary directory and read back with matplotlib's image reader.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Build a controlled cube and a Ploter

We synthesise a `(n_elev, H, W)` GT cube from a spatially varying two-Gaussian mixture and a prediction that is the GT plus mild noise. A parameter cube holds the GT mixture parameters so `plot_profiles` can overlay components. The plotter is configured with the same colormaps the pipeline uses.

In [ ]:
import tempfile
import matplotlib.image as mpimg
from pipelines.inference_pipeline.plots import Ploter
from tools.data.gaussians import GaussianReconstructor

n_elev, H, W = 40, 10, 12
x_axis = np.linspace(-15.0, 15.0, n_elev).astype(np.float64)
rng    = np.random.default_rng(0)

yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
mu1 = -6.0 + 4.0 * (xx / (W - 1))
mu2 =  6.0 - 4.0 * (yy / (H - 1))

params_gt = np.zeros((6, H, W), dtype=np.float32)
params_gt[0] = 1.0; params_gt[1] = mu1; params_gt[2] = 2.0
params_gt[3] = 0.6; params_gt[4] = mu2; params_gt[5] = 3.0

pg5 = params_gt.reshape(1, 2, 3, H, W).astype(np.float32)
x5  = x_axis.reshape(1, 1, -1, 1, 1).astype(np.float32)
gt_cube   = GaussianReconstructor.reconstruct_batch(pg5, x5)[0]
pred_cube = gt_cube + 0.03 * rng.standard_normal(gt_cube.shape).astype(np.float32)

out_dir = Path(tempfile.mkdtemp(prefix='nb08_'))
plotter = Ploter(cmap='jet', err_cmap='magma', normalize=True, fig_dpi=110, save_dpi=110)
print('cube shape:', gt_cube.shape, '-> figures to', out_dir)


## Profile overlays for selected pixels

`plot_profiles` writes one figure per pixel, overlaying GT, prediction and the individual Gaussian components, with per-pixel metrics in the title. We pass two pixels and a matching per-pixel metrics dict, then read the first PNG back.

In [ ]:
pixels = np.array([[2, 3], [7, 9]], dtype=np.int32)
pixel_metrics = {
    'mse': ((pred_cube - gt_cube) ** 2).mean(axis=0),
    'mae': np.abs(pred_cube - gt_cube).mean(axis=0),
    'r2' : np.full((H, W), 0.98, np.float32),
    'cos': np.full((H, W), 0.99, np.float32),
}

profile_paths = plotter.plot_profiles(
    pred_curves=pred_cube, gt_curves=gt_cube, params_pred=params_gt,
    x_axis=x_axis, pixels=pixels, tag='demo', out_dir=out_dir,
    n_gaussians=2, pixel_metrics=pixel_metrics, az_offset=0, rg_offset=0,
)
print('profile figures written:', len(profile_paths))

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.imshow(mpimg.imread(profile_paths[0])); ax.axis('off')
ax.set_title('plot_profiles output (pixel 0)')
plt.show()


## Tomogram cross-section (range cut)

`plot_tomogram_slice` writes three panels (GT, prediction, absolute error) for a cut along one axis. We take a range cut and display the prediction and error panels.

In [ ]:
slice_paths = plotter.plot_tomogram_slice(
    pred_cube=pred_cube, gt_cube=gt_cube, axis='range', index=W // 2,
    x_axis=x_axis, out_dir=out_dir, stem='range_cut', az_offset=0, rg_offset=0,
    ssim_value=0.97,
)
print('slice panels written:', [p.name for p in slice_paths])

fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.0))
for ax, p in zip(axes, slice_paths):
    ax.imshow(mpimg.imread(p)); ax.axis('off'); ax.set_title(p.stem.split('_')[-1])
fig.tight_layout()
plt.show()


## Elevation-plane slice

`plot_elevation_intensity_slice` writes the GT / prediction / error of a single horizontal `H x W` plane at one elevation bin. We take the central bin.

In [ ]:
elev_paths = plotter.plot_elevation_intensity_slice(
    pred_cube=pred_cube, gt_cube=gt_cube, elev_idx=n_elev // 2,
    x_axis=x_axis, out_dir=out_dir, stem='elev_mid', az_offset=0, rg_offset=0,
    ssim_value=0.96,
)
fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.6))
for ax, p in zip(axes, elev_paths):
    ax.imshow(mpimg.imread(p)); ax.axis('off'); ax.set_title(p.stem.split('_')[-1])
fig.tight_layout()
plt.show()


## Expected visual outcome

The profile figure should show a black GT curve, a red dashed prediction nearly on top of it, and two coloured Gaussian components summing to the total. The range-cut panels should show two diagonal intensity ridges (from the spatially drifting means) with a faint error panel. The elevation plane should show smooth GT and prediction maps with a low-magnitude error map, confirming each slice routine renders correctly oriented, low-error outputs.